# Cleaning Missing Value

---

## Apa itu Missing Value?

**Missing value** (nilai yang hilang) adalah kondisi di mana suatu kolom dalam dataset tidak memiliki nilai atau datanya kosong. Dalam pandas, missing value biasanya direpresentasikan sebagai:

- `NaN` *(Not a Number)* untuk data numerik
- `None` untuk data objek/string
- `NaT` *(Not a Time)* untuk data waktu/datetime

---

## Mengapa Missing Value Perlu Ditangani?

| Masalah | Dampak |
|---|---|
| Data tidak lengkap | Analisis menjadi bias |
| Model Machine Learning error | Banyak algoritma tidak bisa memproses NaN |
| Statistik tidak akurat | Mean, median, dan modus terdistorsi |

---

## Strategi Menangani Missing Value

1. **Deteksi** — Cari tahu seberapa banyak data yang hilang
2. **Hapus** — Buang baris/kolom yang mengandung NaN menggunakan `dropna()`
3. **Imputasi** — Isi nilai yang hilang dengan nilai tertentu menggunakan `fillna()`

---

> Dataset yang digunakan: `Building_Permits.csv` — dataset izin bangunan yang mengandung banyak kolom dengan missing value.

## 1. Import Library

Kita membutuhkan dua library utama:

- **`pandas`** — untuk manipulasi dan analisis data tabular (DataFrame)
- **`numpy`** — untuk operasi numerik dan matematika tingkat tinggi

In [ ]:
import pandas as pd
import numpy as np

## 2. Load Dataset

Kita membaca file CSV `Building_Permits.csv` menggunakan `pd.read_csv()`.

Fungsi ini membaca file dan langsung menyimpannya dalam bentuk **DataFrame** yaitu struktur data dua dimensi seperti tabel spreadsheet.

> Pastikan path file sesuai dengan lokasi file CSV di sistem kamu.

In [ ]:
DataDicBuild = pd.read_csv("Data Cleaning/Building_Permits.csv")

np.random.seed(0)

## 3. Eksplorasi Awal Data

Sebelum membersihkan data, kita harus mengenal dataset terlebih dahulu.

Metode `.head()` menampilkan **5 baris pertama** dari DataFrame, berguna untuk melihat struktur kolom dan contoh datanya.

In [ ]:
print("===== 5 DATA BARIS PERTAMA =====")
print(DataDicBuild.head())

## 4. Mendeteksi dan Menghitung Missing Value

Langkah penting sebelum menangani missing value adalah **mengukur seberapa besar** proporsi data yang hilang.

### Metode yang digunakan:

| Fungsi | Kegunaan |
|---|---|
| `.isnull()` | Menghasilkan DataFrame boolean: True jika NaN, False jika tidak |
| `.sum()` | Menjumlahkan True (= 1) per kolom, total NaN per kolom |
| `np.prod(df.shape)` | Menghitung total keseluruhan sel = baris x kolom |

### Formula Persentase Missing Value:

```
Persentase Missing = (Total NaN Seluruh Kolom / Total Seluruh Sel) x 100%
```

In [ ]:
# Hitung jumlah missing value per kolom
total_hilang_perkolom = DataDicBuild.isnull().sum()

# Hitung total keseluruhan sel dalam DataFrame (baris x kolom)
total_keseluruhan = np.prod(DataDicBuild.shape)

# Hitung total missing value dari seluruh kolom
total_hilang_semua = total_hilang_perkolom.sum()

# Hitung persentase data yang hilang
percentage_missing = (total_hilang_semua / total_keseluruhan) * 100

print("===== persentase data yang value-hilang adalah =====")
print(percentage_missing)

### Detail Missing Value Per Kolom

Mari kita lihat kolom mana saja yang memiliki missing value terbanyak:

In [ ]:
# Tampilkan jumlah missing value per kolom (diurutkan terbesar)
missing_summary = total_hilang_perkolom[total_hilang_perkolom > 0].sort_values(ascending=False)
print(f"Jumlah kolom dengan missing value: {len(missing_summary)}")
print("\nTop 10 kolom dengan missing value terbanyak:")
print(missing_summary.head(10))

## 5. Menghapus Kolom dengan Missing Value menggunakan dropna()

Salah satu cara menangani missing value adalah **menghapus** kolom yang memiliki minimal satu NaN.

### Sintaks:
```python
df.dropna(axis=1)
```

### Parameter penting:

| Parameter | Nilai | Keterangan |
|---|---|---|
| `axis` | `0` (default) | Hapus baris yang mengandung NaN |
| `axis` | `1` | Hapus kolom yang mengandung NaN |
| `how` | `any` (default) | Hapus jika ada satu saja NaN |
| `how` | `all` | Hapus hanya jika semua nilai NaN |
| `thresh` | angka | Hapus jika jumlah non-NaN kurang dari thresh |

> **Kekurangan:** Metode ini membuang kolom secara keseluruhan — informasi yang masih valid di kolom tersebut ikut hilang. Gunakan hanya jika kolom memang mayoritas kosong.

In [ ]:
# Hapus semua kolom yang memiliki minimal 1 NaN
DataDicBuild_with_na_dropped = DataDicBuild.dropna(axis=1)

print("===== MENGHAPUS KOLOM YANG MINIMAL ADA SATU NaN =====")
print(DataDicBuild_with_na_dropped)

## 6. Perbandingan Jumlah Kolom Sebelum dan Sesudah dropna()

Kita bisa mengukur **berapa banyak kolom yang tereliminasi** karena mengandung missing value.

Properti `.shape` mengembalikan tuple `(jumlah_baris, jumlah_kolom)` dari DataFrame:

- `.shape[0]` — jumlah baris
- `.shape[1]` — jumlah kolom

In [ ]:
data = DataDicBuild.shape[1]
data_dropped = DataDicBuild_with_na_dropped.shape[1]
dropped_columns = data - data_dropped

print("===== SELISIH KOLOM ADA KEKOSONGAN DAN KOLOM FULL =====")
print(dropped_columns)

## 7. Mengisi Missing Value menggunakan fillna() dengan bfill

Alternatif yang lebih baik dari menghapus adalah **mengisi (imputasi)** nilai yang hilang.

### Metode Imputasi yang Umum:

| Metode | Keterangan |
|---|---|
| `fillna(nilai)` | Isi NaN dengan nilai tetap, misalnya 0 |
| `fillna(df.mean())` | Isi NaN dengan rata-rata kolom |
| `fillna(method='ffill')` | Forward fill — salin nilai dari baris sebelumnya |
| `fillna(method='bfill')` | Backward fill — salin nilai dari baris berikutnya |

### Penjelasan kode:

```python
DataDicBuild.fillna(method='bfill', axis=0).fillna(0)
```

**Langkah 1** — `fillna(method='bfill', axis=0)`:  
Isi NaN dengan nilai dari **baris berikutnya** (backward fill dari bawah ke atas).

**Langkah 2** — `.fillna(0)`:  
Jika masih ada NaN yang tersisa (misalnya di baris terakhir yang tidak punya referensi bfill), isi dengan **angka 0**.

> **Chaining** dua `fillna()` memastikan tidak ada NaN yang tersisa setelah proses imputasi.

In [ ]:
# Imputasi: backward fill dulu, lalu isi sisa NaN dengan 0
Data_with_Na_inputed = DataDicBuild.fillna(method='bfill', axis=0).fillna(0)

print("===== DATA SUDAH TIDAK KOSONG, BISA DILIAT =====")
print(Data_with_Na_inputed)

## 8. Verifikasi Persentase Missing Value Setelah Imputasi

Langkah terakhir adalah **memverifikasi** bahwa tidak ada missing value yang tersisa setelah proses imputasi.

Persentase yang diharapkan adalah **0%** — artinya data sudah bersih dan siap digunakan untuk analisis atau pemodelan.

In [ ]:
# Hitung ulang missing value setelah imputasi
total_hilang_baru = Data_with_Na_inputed.isnull().sum()
total_seluruh = np.prod(Data_with_Na_inputed.shape)
total_semuahilang = total_hilang_baru.sum()

persentase = (total_semuahilang / total_seluruh) * 100

print("===== PERSENTASE DATA HILANG DENGAN TOTAL DATA")
print(persentase)

## 9. Ringkasan dan Kesimpulan

---

### Alur Kerja Data Cleaning Missing Value

```
Load Data
    |
Deteksi Missing Value (.isnull().sum())
    |
Hitung Persentase Missing
    |
    |--- Hapus Kolom dengan NaN (.dropna(axis=1))
    |        |
    |    Bandingkan jumlah kolom sebelum dan sesudah
    |
    |--- Imputasi NaN (.fillna(method='bfill').fillna(0))
             |
         Verifikasi persentase NaN = 0%
```

---

### Fungsi-Fungsi Utama yang Dipelajari

| Fungsi | Tujuan |
|---|---|
| `pd.read_csv()` | Membaca file CSV ke dalam DataFrame |
| `.isnull().sum()` | Menghitung jumlah missing value per kolom |
| `np.prod(df.shape)` | Menghitung total sel dalam DataFrame |
| `.dropna(axis=1)` | Menghapus kolom yang memiliki NaN |
| `.shape[1]` | Mengambil jumlah kolom DataFrame |
| `.fillna(method='bfill')` | Mengisi NaN dengan backward fill |
| `.fillna(0)` | Mengisi sisa NaN dengan nilai 0 |

---

### Kapan Gunakan Hapus vs Imputasi?

| Situasi | Rekomendasi |
|---|---|
| Kolom lebih dari 70-80% NaN | Hapus kolom dengan dropna |
| Data time-series atau berurutan | Gunakan ffill atau bfill |
| Data numerik acak | Isi dengan mean() atau median() |
| Data kategorikal | Isi dengan modus atau Unknown |
| Sangat sedikit NaN | Hapus baris dengan dropna(axis=0) |

---

> **Kunci sukses data cleaning:** Pahami karakteristik data sebelum memilih strategi penanganan missing value. Tidak ada satu metode yang cocok untuk semua kasus!